# CBRAIN OASIS Model Comparison (Step 3B)

This notebook trains one stronger comparison model under frozen Step 2 contracts and compares it against the Step 3A logistic baseline.

Environment note: run with project venv (`.venv`) that includes scikit-learn.

In [ ]:
# 1) Load frozen artifacts and baseline metrics
from __future__ import annotations

import csv
import json
from datetime import date
from pathlib import Path

import numpy as np

try:
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import (
        accuracy_score,
        average_precision_score,
        confusion_matrix,
        f1_score,
        precision_score,
        recall_score,
        roc_auc_score,
    )
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError('scikit-learn is required for Step 3B. Use .venv/bin/python.') from exc

cwd = Path.cwd().resolve()
candidate_roots = [cwd]
if cwd.name == 'notebooks':
    candidate_roots.append(cwd.parent)
else:
    candidate_roots.append(cwd / 'projects' / 'cbrain_oasis_cognition')
    candidate_roots.append(cwd.parent)

repo_root = None
for candidate in candidate_roots:
    if (candidate / 'results' / 'preprocessing' / 'X_train.csv').exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError('Could not locate preprocessing artifacts from current working directory.')

pre_dir = repo_root / 'results' / 'preprocessing'
cohort_dir = repo_root / 'results' / 'cohort'
model_dir = repo_root / 'results' / 'modeling'
docs_path = repo_root / 'docs' / 'model_comparison_step3b.md'

x_train_path = pre_dir / 'X_train.csv'
x_validation_path = pre_dir / 'X_validation.csv'
y_train_path = pre_dir / 'y_train.csv'
y_validation_path = pre_dir / 'y_validation.csv'
feature_manifest_path = pre_dir / 'feature_manifest.json'
split_subjects_path = cohort_dir / 'split_subjects.json'
baseline_metrics_path = model_dir / 'baseline_metrics.json'

feature_manifest = json.loads(feature_manifest_path.read_text(encoding='utf-8'))
split_payload = json.loads(split_subjects_path.read_text(encoding='utf-8'))
baseline_metrics = json.loads(baseline_metrics_path.read_text(encoding='utf-8'))

def _read_csv(path: Path) -> tuple[list[str], list[dict]]:
    with path.open('r', encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f)
        return list(reader.fieldnames or []), [dict(r) for r in reader]

x_train_cols, x_train_rows = _read_csv(x_train_path)
x_validation_cols, x_validation_rows = _read_csv(x_validation_path)
y_train_cols, y_train_rows = _read_csv(y_train_path)
y_validation_cols, y_validation_rows = _read_csv(y_validation_path)

print(f'Repo root: {repo_root}')
print(f'Train rows: {len(x_train_rows)}; Validation rows: {len(x_validation_rows)}')

In [ ]:
# 2) Leakage and contract gates
expected_feature_order = feature_manifest['final_feature_order']
if x_train_cols != expected_feature_order:
    raise AssertionError(f'X_train column mismatch: {x_train_cols} vs {expected_feature_order}')
if x_validation_cols != expected_feature_order:
    raise AssertionError(f'X_validation column mismatch: {x_validation_cols} vs {expected_feature_order}')

if y_train_cols != ['cognitive_impairment'] or y_validation_cols != ['cognitive_impairment']:
    raise AssertionError('Label files must contain exactly cognitive_impairment.')

if len(x_train_rows) != len(y_train_rows) or len(x_validation_rows) != len(y_validation_rows):
    raise AssertionError('X and y row-count mismatch.')

excluded = {'CDR','Group','MMSE','Subject.ID','MRI.ID','rownames','Hand','Visit','MR.Delay'}
if any(c in excluded for c in x_train_cols) or any(c in excluded for c in x_validation_cols):
    raise AssertionError('Leakage fields found in model matrices.')

if split_payload['overlap_count'] != 0:
    raise AssertionError('Split leakage gate failed: overlap_count != 0')

train_subject_ids = sorted(split_payload['train_subject_ids'])
validation_subject_ids = sorted(split_payload['validation_subject_ids'])
if len(train_subject_ids) != len(x_train_rows) or len(validation_subject_ids) != len(x_validation_rows):
    raise AssertionError('Split subject counts do not align with matrix rows.')

if baseline_metrics.get('feature_order') != expected_feature_order:
    raise AssertionError('Baseline feature order differs from frozen manifest.')

print('Leakage/contract gates: PASS')

In [ ]:
# 3) Build matrices
def _rows_to_matrix(rows: list[dict], columns: list[str]) -> np.ndarray:
    return np.array([[float(r[c]) for c in columns] for r in rows], dtype=float)

def _rows_to_label(rows: list[dict]) -> np.ndarray:
    return np.array([int(r['cognitive_impairment']) for r in rows], dtype=int)

X_train = _rows_to_matrix(x_train_rows, expected_feature_order)
X_validation = _rows_to_matrix(x_validation_rows, expected_feature_order)
y_train = _rows_to_label(y_train_rows)
y_validation = _rows_to_label(y_validation_rows)

if np.isnan(X_train).any() or np.isnan(X_validation).any():
    raise AssertionError('NaN detected in features.')
print('Shapes:', X_train.shape, X_validation.shape)

In [ ]:
# 4) Train stronger comparison model: RandomForestClassifier
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42,
    n_jobs=1,
)
rf.fit(X_train, y_train)

rf_train_prob = rf.predict_proba(X_train)[:, 1]
rf_validation_prob = rf.predict_proba(X_validation)[:, 1]
rf_train_pred = (rf_train_prob >= 0.5).astype(int)
rf_validation_pred = (rf_validation_prob >= 0.5).astype(int)

rf_info = {
    'n_estimators': 500,
    'max_depth': None,
    'min_samples_leaf': 2,
    'class_weight': 'balanced',
    'random_state': 42,
    'n_jobs': 1,
}
print('RandomForest trained with', rf_info)

In [ ]:
# 5) Evaluate and compare against Step 3A baseline
def _metrics(y_true: np.ndarray, y_prob: np.ndarray, y_pred: np.ndarray) -> dict:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = [int(x) for x in cm.ravel()]
    return {
        'threshold': 0.5,
        'roc_auc': float(roc_auc_score(y_true, y_prob)),
        'pr_auc': float(average_precision_score(y_true, y_prob)),
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        'specificity': float(tn / (tn + fp) if (tn + fp) else 0.0),
        'confusion_matrix': {'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn},
    }

rf_train_metrics = _metrics(y_train, rf_train_prob, rf_train_pred)
rf_validation_metrics = _metrics(y_validation, rf_validation_prob, rf_validation_pred)

baseline_val = baseline_metrics['validation_metrics']
comparison = {
    'baseline_model': baseline_metrics['model_name'],
    'comparison_model': 'random_forest_sklearn',
    'validation_baseline': baseline_val,
    'validation_comparison': rf_validation_metrics,
    'validation_delta': {
        'roc_auc': float(rf_validation_metrics['roc_auc'] - baseline_val['roc_auc']),
        'pr_auc': float(rf_validation_metrics['pr_auc'] - baseline_val['pr_auc']),
        'accuracy': float(rf_validation_metrics['accuracy'] - baseline_val['accuracy']),
        'f1': float(rf_validation_metrics['f1'] - baseline_val['f1']),
    },
}

print('Baseline val ROC-AUC:', round(baseline_val['roc_auc'], 4))
print('RF val ROC-AUC      :', round(rf_validation_metrics['roc_auc'], 4))
print('Delta ROC-AUC       :', round(comparison['validation_delta']['roc_auc'], 4))

In [ ]:
# 6) Materialize Step 3B artifacts + docs
model_dir.mkdir(parents=True, exist_ok=True)

rf_metrics_path = model_dir / 'step3b_random_forest_metrics.json'
rf_val_pred_path = model_dir / 'step3b_random_forest_validation_predictions.csv'
rf_feat_imp_path = model_dir / 'step3b_random_forest_feature_importance.csv'
comparison_path = model_dir / 'model_comparison.json'

rf_metrics_payload = {
    'model_name': 'random_forest_sklearn',
    'date': date.today().isoformat(),
    'hyperparameters': rf_info,
    'feature_order': expected_feature_order,
    'train_metrics': rf_train_metrics,
    'validation_metrics': rf_validation_metrics,
    'train_row_count': int(X_train.shape[0]),
    'validation_row_count': int(X_validation.shape[0]),
}
rf_metrics_path.write_text(json.dumps(rf_metrics_payload, indent=2, sort_keys=True) + '\n', encoding='utf-8')

comparison_path.write_text(json.dumps(comparison, indent=2, sort_keys=True) + '\n', encoding='utf-8')

with rf_val_pred_path.open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['Subject.ID','y_true','y_probability','y_pred_threshold_0_5'])
    writer.writeheader()
    for sid, y_true, y_prob, y_hat in zip(validation_subject_ids, y_validation.tolist(), rf_validation_prob.tolist(), rf_validation_pred.tolist()):
        writer.writerow({
            'Subject.ID': sid,
            'y_true': int(y_true),
            'y_probability': float(y_prob),
            'y_pred_threshold_0_5': int(y_hat),
        })

with rf_feat_imp_path.open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['feature','importance'])
    writer.writeheader()
    pairs = sorted(zip(expected_feature_order, rf.feature_importances_.tolist()), key=lambda x: x[1], reverse=True)
    for feat, imp in pairs:
        writer.writerow({'feature': feat, 'importance': float(imp)})

run_date = date.today().isoformat()
docs_text = f"""# Model Comparison (Step 3B)

Date: {run_date}  
Notebook: `notebooks/05_model_comparison_step3b.ipynb`

## Scope

Compare Step 3A baseline logistic model against one stronger model (`RandomForestClassifier`) **without changing split or preprocessing contracts**.

## Leakage Controls

- Inputs only from frozen Step 2 artifacts (`results/preprocessing/*`, `results/cohort/split_subjects.json`).
- Feature order locked to manifest.
- No subject overlap (`overlap_count=0`).
- No leakage fields present in model matrix.

## Models Compared

- Baseline: `baseline_logistic_sklearn` (from Step 3A artifact).
- Stronger model: `RandomForestClassifier(n_estimators=500, min_samples_leaf=2, class_weight=balanced, random_state=42)`.

## Validation Metrics

| Metric | Baseline Logistic | Random Forest | Delta (RF - Baseline) |
|---|---:|---:|---:|
| ROC-AUC | {baseline_val["roc_auc"]:.4f} | {rf_validation_metrics["roc_auc"]:.4f} | {comparison["validation_delta"]["roc_auc"]:.4f} |
| PR-AUC | {baseline_val["pr_auc"]:.4f} | {rf_validation_metrics["pr_auc"]:.4f} | {comparison["validation_delta"]["pr_auc"]:.4f} |
| Accuracy | {baseline_val["accuracy"]:.4f} | {rf_validation_metrics["accuracy"]:.4f} | {comparison["validation_delta"]["accuracy"]:.4f} |
| F1 | {baseline_val["f1"]:.4f} | {rf_validation_metrics["f1"]:.4f} | {comparison["validation_delta"]["f1"]:.4f} |

Validation confusion matrix (Random Forest):
- TP={rf_validation_metrics["confusion_matrix"]["tp"]}, TN={rf_validation_metrics["confusion_matrix"]["tn"]}, FP={rf_validation_metrics["confusion_matrix"]["fp"]}, FN={rf_validation_metrics["confusion_matrix"]["fn"]}

## Step 3B Artifacts

- `results/modeling/step3b_random_forest_metrics.json`
- `results/modeling/step3b_random_forest_validation_predictions.csv`
- `results/modeling/step3b_random_forest_feature_importance.csv`
- `results/modeling/model_comparison.json`
"""
docs_path.write_text(docs_text, encoding='utf-8')

print('Wrote:', rf_metrics_path)
print('Wrote:', rf_val_pred_path)
print('Wrote:', rf_feat_imp_path)
print('Wrote:', comparison_path)
print('Wrote:', docs_path)